# 03 — Operations

Hands-on companion to [`docs/03-operations.md`](../docs/03-operations.md).

Operations turn individual `Leaf` values into useful expressions. We will look at the small rules behind `-`, `/`, and `@`, then combine them into one familiar layer.

> Run this with the notebook extra installed: `pip install -e ".[notebooks]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bonsaigrad import Leaf
from notebooks.utils import show_array

## Derived elementwise operations

Subtraction and division are built from the primitive operations already available:

```python
a - b  # a + (b * -1)
a / b  # a * (b ** -1)
```

That means their gradients follow from the same small set of rules.

In [ ]:
a, b = Leaf(3.0), Leaf(2.0)
difference = a - b
quotient = a / b

difference.wire()
print(f"a - b = {difference.data.item():g}   →   da = {a.grad.item():g}, db = {b.grad.item():g}")

a.rest()
b.rest()
quotient.wire()
print(f"a / b = {quotient.data.item():g}   →   da = {a.grad.item():g}, db = {b.grad.item():g}")

np.testing.assert_allclose([a.grad, b.grad], [1 / b.data, -a.data / b.data ** 2])
print("Both results come from +, *, and ** — no separate backward rule is needed.")

## Matrix multiplication changes axes

Elementwise operations keep every axis. Matrix multiplication joins the shared axis instead: `(2, 3) @ (3, 2) → (2, 2)`. During the backward pass, transposes put the missing axis back.

In [ ]:
x = Leaf([[1.0, 2.0, -1.0], [0.5, -2.0, 3.0]])
w = Leaf([[1.0, -1.0], [2.0, 0.5], [-0.5, 1.5]])
out = x @ w
out.wire()

fig, axes = plt.subplots(2, 3, figsize=(9, 5))
show_array(axes[0, 0], x.data, "inputs x")
show_array(axes[0, 1], w.data, "weights w")
show_array(axes[0, 2], out.data, "out = x @ w")
show_array(axes[1, 0], out.grad, "seeded out.grad", cmap="Reds")
show_array(axes[1, 1], x.grad, "x.grad = out.grad @ w.T", cmap="Reds")
show_array(axes[1, 2], w.grad, "w.grad = x.T @ out.grad", cmap="Reds")
plt.tight_layout()
plt.show()

np.testing.assert_allclose(x.grad, out.grad @ w.data.T)
np.testing.assert_allclose(w.grad, x.data.T @ out.grad)
print(f"{x.data.shape} @ {w.data.shape} → {out.data.shape}; gradients return to {x.grad.shape} and {w.grad.shape}.")

## One expression, many examples

`x @ w + b` is a linear layer written directly with `Leaf`s. `w` and `b` are shared by every row of `x`, so their gradients collect contributions across the batch.

In [ ]:
x = Leaf([[1.0, 2.0, -1.0], [0.5, -2.0, 3.0]])
w = Leaf([[1.0, -1.0], [2.0, 0.5], [-0.5, 1.5]])
b = Leaf([0.25, -0.75])
scores = x @ w + b
scores.wire()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_array(axes[0], x.data, "batch x")
show_array(axes[1], scores.data, "scores = x @ w + b")
show_array(axes[2], w.grad, "shared w.grad", cmap="Reds")
plt.tight_layout()
plt.show()

np.testing.assert_allclose(b.grad, [2.0, 2.0])
np.testing.assert_allclose(w.grad, x.data.T @ np.ones_like(scores.data))
print(f"b.grad = {b.grad.tolist()}: one contribution from each of the {x.data.shape[0]} examples.")

Elementwise operations broadcast, `@` contracts, and every backward step restores the gradient shape of the value it belongs to. Next: layers built from these pieces.